# Rusted Spire — RL Experiments

Ironclad (Strike × 5, Defend × 4, Bash × 1) vs various Act 1 enemies.  
Algorithm: MaskablePPO (sb3-contrib).  
Environment: Rust headless simulator exposed via PyO3.

Available enemies: `JawWorm`, `Cultist`, `LouseNormal`, `LouseDefensive`, `GremlinNob`  
Default: `Cultist` at Ascension 7.

**Interface v3:** 178 observations and 61 actions with explicit enemy targeting. Old checkpoints and historical metrics require retraining/revalidation. See `docs/RL_INTERFACE.md` and `docs/VALIDATION.md`.

Two environment wrappers:
- **`SpireEnv`** — single combat; pass `enemy=` and `ascension=` to constructor.
- **`MultiCombatEnv`** — chained combats; pass `enemies=[...]` list (one name per slot).

## 1. Imports & sanity check

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

import rusted_spire

print(f"Obs size:    {rusted_spire.OBS_SIZE}")
print(f"Action size: {rusted_spire.ACTION_SIZE}")

## 2. Gymnasium environment

In [ ]:
from spire_env import SpireEnv


## 2b. Multi-combat environment

Chain fights in one episode. HP carries over; block and powers reset between fights.  
Pass `enemies=["Cultist", "GremlinNob"]` to set a fixed sequence, or a single-element list to repeat.  
Reward accumulates: each victory gives `+1 + hp_ratio`, defeat gives `-1`.  
Max episode reward = `len(enemies) × 2.0` (win everything at full HP).

In [ ]:
class MultiCombatEnv(gym.Env):
    """Chain fights, carrying HP; each fight uses the shared versioned wrapper."""

    metadata = {"render_modes": []}

    def __init__(self, enemies=None, ascension=7, max_steps=1000):
        super().__init__()
        self._envs = [
            SpireEnv(enemy=e, ascension=ascension, max_steps=max_steps)
            for e in (enemies or ["Cultist"] * 3)
        ]
        self.observation_space = self._envs[0].observation_space
        self.action_space = self._envs[0].action_space
        self._needs_reset = True
        self._combat_idx = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._base_seed = int(self.np_random.integers(0, 2**32))
        self._combat_idx = 0
        self._needs_reset = False
        return self._envs[0].reset(seed=self._base_seed, options=options)

    def step(self, action):
        if self._needs_reset:
            raise RuntimeError("call reset() before a new chain")
        obs, reward, terminated, truncated, info = self._envs[self._combat_idx].step(action)
        if truncated or (terminated and reward < 0):
            self._needs_reset = True
            return obs, reward, terminated, truncated, info
        if not terminated:
            return obs, reward, False, False, info
        self._combat_idx += 1
        if self._combat_idx == len(self._envs):
            self._needs_reset = True
            return obs, reward, True, False, info
        hp = max(1, round(float(obs[0]) * rusted_spire.MAX_HP))
        obs, info = self._envs[self._combat_idx].reset(
            seed=(self._base_seed + self._combat_idx * 100_003) % 2**32,
            options={"hp": hp},
        )
        return obs, reward, False, False, info

    def action_masks(self):
        if self._needs_reset:
            return np.zeros(rusted_spire.ACTION_SIZE, dtype=bool)
        return self._envs[self._combat_idx].action_masks()

    def close(self):
        for env in self._envs:
            env.close()


## 3. Environment smoke test

Run one episode with random legal actions and print the trajectory.

In [ ]:
env = SpireEnv()
obs, _ = env.reset(seed=42)

total_reward = 0
steps = 0
done = False

while not done:
    mask = env.action_masks()
    legal = np.where(mask)[0]
    action = int(np.random.choice(legal))
    obs, reward, done, truncated, _ = env.step(action)
    done = done or truncated
    total_reward += reward
    steps += 1

print(f"Episode ended in {steps} steps  |  total reward: {total_reward:.2f}")

## 4. Random policy baseline

Win rate of a policy that always picks a random legal action.

In [ ]:
N = 1000

def random_winrate(env_cls, n=N, **kwargs):
    env = env_cls(**kwargs)
    wins, total_rewards = 0, []
    for i in range(n):
        obs, _ = env.reset(seed=i)
        done, ep_reward = False, 0.0
        while not done:
            legal = np.where(env.action_masks())[0]
            obs, reward, done, truncated, _ = env.step(int(np.random.choice(legal)))
            done = done or truncated
            ep_reward += reward
        if reward > 0 and not truncated:
            wins += 1
        total_rewards.append(ep_reward)
    return wins / n, np.mean(total_rewards)

wr1, mr1 = random_winrate(SpireEnv)
print(f"Single combat  — win rate: {wr1:.1%}  mean reward: {mr1:.3f}")

wr3, mr3 = random_winrate(MultiCombatEnv, enemies=["Cultist"] * 3)
print(f"3-combat chain — win rate: {wr3:.1%}  mean reward: {mr3:.3f}  (max possible: 6.0)")

## 5. Train — MaskablePPO

Tweak hyperparameters here before running.

In [ ]:
# --- Hyperparameters ---
TIMESTEPS   = 500_000
N_ENVS      = 8
LEARNING_RATE = 3e-4
N_STEPS     = 512
BATCH_SIZE  = 256
N_EPOCHS    = 10
GAMMA       = 0.99

vec_env  = make_vec_env(SpireEnv, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
eval_env = make_vec_env(SpireEnv, n_envs=4,      vec_env_cls=SubprocVecEnv)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    verbose=1,
    tensorboard_log="runs/",
)

model.learn(total_timesteps=TIMESTEPS)
model.save("models/notebook_v3")
print("Saved to models/notebook_v3")

## 6. Evaluate trained model

In [ ]:
# Load from disk if needed:
# model = MaskablePPO.load("models/notebook_v3", env=SpireEnv())

mean_reward, std_reward = evaluate_policy(
    model,
    SpireEnv(),
    n_eval_episodes=500,
    warn=False,
)
print(f"Mean reward: {mean_reward:.3f} ± {std_reward:.3f}")

# Win rate: reward > 0 means victory
wins = 0
N_EVAL = 500
eval_single = SpireEnv()
for i in range(N_EVAL):
    obs, _ = eval_single.reset(seed=10_000 + i)
    done = False
    while not done:
        action, _ = model.predict(obs, action_masks=eval_single.action_masks(), deterministic=True)
        obs, reward, done, truncated, _ = eval_single.step(action)
        done = done or truncated
        if done and reward > 0:
            wins += 1

print(f"Trained policy win rate: {wins}/{N_EVAL} = {wins/N_EVAL:.1%}")

## 7. Watch one episode

In [ ]:
# Interface offsets are exported by the extension; do not hardcode v1 offsets.
env = SpireEnv()
obs, _ = env.reset(seed=999)
done = False
while not done:
    hand = [
        round(float(obs[rusted_spire.PLAYER_FEATURES + i * rusted_spire.HAND_FEATURES])
              * rusted_spire.CARD_COUNT)
        for i in range(rusted_spire.MAX_HAND)
    ]
    print(f"HP {obs[0] * 80:.0f}, block {obs[1] * 100:.0f}, energy {obs[2] * 3:.0f}; card IDs {hand}")
    for slot in range(rusted_spire.MAX_ENEMIES):
        offset = rusted_spire.ENEMY_OFFSET + slot * rusted_spire.ENEMY_FEATURES
        if obs[offset] == 0:
            continue
        enemy_hp = obs[offset + 2] * obs[offset + 3] * 300
        damage = obs[offset + 6] * 20
        hits = obs[offset + 7] * 4
        print(f"  Target {slot}: HP {enemy_hp:.0f}, attack {damage:.0f} x {hits:.0f}")
    action, _ = model.predict(obs, action_masks=env.action_masks(), deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(int(action))
    done = terminated or truncated
print("TRUNCATED" if truncated else ("WIN" if reward > 0 else "LOSS"))
env.close()
